In [1]:
!pip install pyspark py4j

# Анализ активности онлайн-пользователей

Создание сессиим

In [80]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
.appName("user_activity") \
.getOrCreate()

1.1. Создайте DataFrame df_user_activity со следующей структурой и данными:

In [81]:
from datetime import date

data = [
    (101, date(2025, 1, 1), {"mobile": 3, "desktop": 1}, ["/home", "/products", "/cart"], 4.5),
    (102, date(2025, 1, 1), {"desktop": 2}, ["/home", "/about"], 3.0),
    (101, date(2025, 1, 2), {"mobile": 2}, ["/products", "/checkout"], None),
    (103, date(2025, 1, 2), {"tablet": 1, "mobile": 1}, ["/blog", "/contact"], 5.0),
    (104, date(2025, 1, 3), {"desktop": 4}, ["/dashboard"], 3.5),
    (101, date(2025, 1, 3), {"mobile": 1, "desktop": 1}, ["/home", "/products"], 4.0),
    (105, date(2025, 1, 4), {"mobile": 5}, ["/faq"], None),
    (102, date(2025, 1, 4), {"desktop": 1, "mobile": 1}, ["/settings"], 3.8),
    (103, date(2025, 1, 5), {"tablet": 2}, ["/products"], 4.2),
    (106, date(2025, 1, 5), {"desktop": 3, "mobile": 2}, ["/login", "/profile", "/home"], 4.7),
    (101, date(2025, 1, 6), {"mobile": 1}, ["/cart", "/checkout"], 4.0),
    (104, date(2025, 1, 6), {"desktop": 2, "tablet": 1}, ["/contact"], None),
    (105, date(2025, 1, 7), {"mobile": 3, "desktop": 1}, ["/pricing"], 4.1),
    (106, date(2025, 1, 7), {"desktop": 1}, ["/home", "/about"], 3.9),
    (107, date(2025, 1, 8), {"mobile": 4, "tablet": 2}, ["/products", "/blog"], 4.9)
]

df_user_activity_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("date", DateType(), True),
    StructField("device_type", MapType(StringType(), IntegerType()), True),
    StructField("page_views", ArrayType(StringType()), True),
    StructField("rating", DoubleType(), True)
])

df_user_activity = spark.createDataFrame(data, df_user_activity_schema)
df_user_activity.show()

+-------+----------+--------------------+--------------------+------+
|user_id|      date|         device_type|          page_views|rating|
+-------+----------+--------------------+--------------------+------+
|    101|2025-01-01|{mobile -> 3, des...|[/home, /products...|   4.5|
|    102|2025-01-01|      {desktop -> 2}|     [/home, /about]|   3.0|
|    101|2025-01-02|       {mobile -> 2}|[/products, /chec...|  NULL|
|    103|2025-01-02|{mobile -> 1, tab...|   [/blog, /contact]|   5.0|
|    104|2025-01-03|      {desktop -> 4}|        [/dashboard]|   3.5|
|    101|2025-01-03|{mobile -> 1, des...|  [/home, /products]|   4.0|
|    105|2025-01-04|       {mobile -> 5}|              [/faq]|  NULL|
|    102|2025-01-04|{mobile -> 1, des...|         [/settings]|   3.8|
|    103|2025-01-05|       {tablet -> 2}|         [/products]|   4.2|
|    106|2025-01-05|{mobile -> 2, des...|[/login, /profile...|   4.7|
|    101|2025-01-06|       {mobile -> 1}|  [/cart, /checkout]|   4.0|
|    104|2025-01-06|

1.2. Выведите схему созданного DataFrame.

In [82]:
df_user_activity.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- device_type: map (nullable = true)
 |    |-- key: string
 |    |-- value: integer (valueContainsNull = true)
 |-- page_views: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rating: double (nullable = true)



2.1. Рассчитайте total_sessions_count для каждой записи (строки): Создайте новую колонку total_sessions_count. Значение этой колонки должно быть суммой количества сессий по всем устройствам (mobile, desktop, tablet) для каждой записи. \
2.2. Извлеките количество мобильных сессий: Создайте новую колонку mobile_sessions, которая будет содержать количество сессий, проведенных с "mobile" устройства из sessions_by_device. Если данных о "mobile" устройстве нет в данных для данной записи, значение должно быть 0.

In [83]:
df_user_activity_total_sessions_cnt = (
    df_user_activity
    .withColumn("mobile_sessions",  F.coalesce(F.col("device_type.mobile"),  F.lit(0)))
    .withColumn("desktop", F.coalesce(F.col("device_type.desktop"), F.lit(0)))
    .withColumn("tablet",  F.coalesce(F.col("device_type.tablet"),  F.lit(0)))
    .withColumn(
        "total_sessions_count",
        F.col("mobile_sessions") + F.col("desktop") + F.col("tablet")
    )
    .drop( "desktop", "tablet")
)
df_user_activity_total_sessions_cnt.show()

+-------+----------+--------------------+--------------------+------+---------------+--------------------+
|user_id|      date|         device_type|          page_views|rating|mobile_sessions|total_sessions_count|
+-------+----------+--------------------+--------------------+------+---------------+--------------------+
|    101|2025-01-01|{mobile -> 3, des...|[/home, /products...|   4.5|              3|                   4|
|    102|2025-01-01|      {desktop -> 2}|     [/home, /about]|   3.0|              0|                   2|
|    101|2025-01-02|       {mobile -> 2}|[/products, /chec...|  NULL|              2|                   2|
|    103|2025-01-02|{mobile -> 1, tab...|   [/blog, /contact]|   5.0|              1|                   2|
|    104|2025-01-03|      {desktop -> 4}|        [/dashboard]|   3.5|              0|                   4|
|    101|2025-01-03|{mobile -> 1, des...|  [/home, /products]|   4.0|              1|                   2|
|    105|2025-01-04|       {mobile ->

3: Агрегация данных по пользователям

3.1. Для каждого пользователя рассчитайте total_sessions_all_time: Это общее количество всех сессий пользователя за весь период наблюдений. Отсортируйте результат по total_sessions_all_time в порядке убывания.

3.2. Для каждого пользователя получите unique_visited_pages_all_time: Это список всех уникальных посещенных страниц пользователя за весь период.

In [95]:
df_user_activity_total_sessions_all_time = (
    df_user_activity_total_sessions_cnt
    .groupBy("user_id")
    .agg(
        F.sum("total_sessions_count").alias("total_sessions_all_time"),
        F.collect_set("page_views").alias("visited_pages_nested")
    )
    .withColumn(
        "unique_visited_pages_all_time",
        F.array_distinct(F.flatten("visited_pages_nested"))
    )
    .drop("visited_pages_nested")
    .orderBy(F.col("total_sessions_all_time").desc())
)

df_user_activity_total_sessions_all_time.show(truncate=False)


+-------+-----------------------+------------------------------------+
|user_id|total_sessions_all_time|unique_visited_pages_all_time       |
+-------+-----------------------+------------------------------------+
|101    |9                      |[/products, /checkout, /cart, /home]|
|105    |9                      |[/faq, /pricing]                    |
|104    |7                      |[/dashboard, /contact]              |
|107    |6                      |[/products, /blog]                  |
|106    |6                      |[/login, /profile, /home, /about]   |
|103    |4                      |[/products, /blog, /contact]        |
|102    |4                      |[/settings, /home, /about]          |
+-------+-----------------------+------------------------------------+



4: Фильтрация данных

4.1. Отфильтруйте DataFrame df_user_activity, чтобы показать только те записи, где usability_rating выше 3.5. Отсортируйте результат по usability_rating в порядке убывания.


In [96]:
df_user_activity_filter = df_user_activity.filter(df_user_activity.rating > 3.5).sort(df_user_activity.rating.desc())
df_user_activity_filter.show()

+-------+----------+--------------------+--------------------+------+
|user_id|      date|         device_type|          page_views|rating|
+-------+----------+--------------------+--------------------+------+
|    103|2025-01-02|{mobile -> 1, tab...|   [/blog, /contact]|   5.0|
|    107|2025-01-08|{mobile -> 4, tab...|  [/products, /blog]|   4.9|
|    106|2025-01-05|{mobile -> 2, des...|[/login, /profile...|   4.7|
|    101|2025-01-01|{mobile -> 3, des...|[/home, /products...|   4.5|
|    103|2025-01-05|       {tablet -> 2}|         [/products]|   4.2|
|    105|2025-01-07|{mobile -> 3, des...|          [/pricing]|   4.1|
|    101|2025-01-03|{mobile -> 1, des...|  [/home, /products]|   4.0|
|    101|2025-01-06|       {mobile -> 1}|  [/cart, /checkout]|   4.0|
|    106|2025-01-07|      {desktop -> 1}|     [/home, /about]|   3.9|
|    102|2025-01-04|{mobile -> 1, des...|         [/settings]|   3.8|
+-------+----------+--------------------+--------------------+------+



In [97]:
spark.stop()

# Анализ финансовых данных АвтоВАЗ

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("auto_vaz") \
    .getOrCreate()

Предварительная подготовка данных:
Приведите все числовые колонки к соответствующим типам (например, DoubleType для финансовых показателей и IntegerType для года). Это обеспечит корректность математических операций.
Стандартизируйте названия столбцов. Убедитесь, что имена столбцов приведены к единообразному, удобному для программной обработки формату (например, нижний регистр, использование нижнего подчеркивания вместо пробелов, удаление специальных символов). Это поможет избежать ошибок при обращении к коло

In [39]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, DoubleType, StringType
)

df_auto = spark.read.csv(
    '/content/drive/MyDrive/py_spark_data/avtovaz.csv',
    header = True,
    inferSchema = True
)
df_auto.printSchema()
new_columns = [c.strip().replace(" ", "_").replace('.', '').lower() for c in df_auto.columns]
# Применяем новые имена к DataFrame.
df_auto = df_auto.toDF(*new_columns)
df_auto.printSchema()

root
 |-- Год: integer (nullable = true)
 |-- Реализация (внутренний рынок: тыс): double (nullable = true)
 |-- Реализация (внешний рынок: тыс): double (nullable = true)
 |-- Себестоимость реализации (млн. руб): double (nullable = true)
 |-- Валовая прибыль от реализации: double (nullable = true)
 |-- Общая сумма дивидендов (млн. руб): double (nullable = true)
 |-- Чистая прибыль отчётного года(млн. руб): integer (nullable = true)
 |-- Всего текущие активы: double (nullable = true)
 |-- Потоки денежных средств от инвестиционной деятельности: double (nullable = true)
 |-- Потоки денежных средств от фин деятельности: double (nullable = true)
 |-- Чистые денежные средства  от операционной деятельности: double (nullable = true)
 |--  Прибыль до налогов: double (nullable = true)
 |--  Тип: string (nullable = true)

root
 |-- год: integer (nullable = true)
 |-- реализация_(внутренний_рынок:_тыс): double (nullable = true)
 |-- реализация_(внешний_рынок:_тыс): double (nullable = true)
 |-- себ

Задание 1: Анализ динамики чистой прибыли и ранжирование по росту
В этом задании Вам предстоит проанализировать годовую динамику чистой прибыли АвтоВАЗ с использованием оконных функций.

1. Рассчитать чистую прибыль за предыдущий год для каждой записи.
2. Вычислить абсолютное изменение чистой прибыли (рост или падение) по сравнению с предыдущим годом.
3. Ранжировать все годы по величине этого абсолютного изменения чистой прибыли, от наибольшего роста к наименьшему.
4. Вывести следующие столбцы для всех годов, отсортировав результат по рангу:
* Год
* Чистая прибыль отчётного года
* Чистая прибыль предыдущего года
* Изменение прибыли (абсолютное значение)
* Ранг роста прибыли

In [66]:
window_spec = Window.orderBy('год')

df_auto_net_profit = df_auto \
                      .withColumn(
                        'чистая_прибыль_предыдущего_года',
                        F.lag('чистая_прибыль_отчётного_года(млн_руб)', 1).over(window_spec)
                      ).withColumn(
                        "изменение_прибыли",
                        F.col('чистая_прибыль_отчётного_года(млн_руб)') -
                        F.col('чистая_прибыль_предыдущего_года')
                      ).withColumn(
                        'ранг_роста_прибыли',
                        F.rank().over(
                            Window.orderBy(
                                F.abs(
                                    F.coalesce(F.col('изменение_прибыли'), F.lit(0))
                                ).desc()
                            )
                        )
    )


df_auto_net_profit.select(
    'год',
    'чистая_прибыль_отчётного_года(млн_руб)',
    'чистая_прибыль_предыдущего_года',
    'изменение_прибыли',
    'ранг_роста_прибыли'
).show(24)

+----+--------------------------------------+-------------------------------+-----------------+------------------+
| год|чистая_прибыль_отчётного_года(млн_руб)|чистая_прибыль_предыдущего_года|изменение_прибыли|ранг_роста_прибыли|
+----+--------------------------------------+-------------------------------+-----------------+------------------+
|2010|                                  1500|                         -38500|            40000|                 1|
|2009|                                -38500|                          -6684|           -31816|                 2|
|2017|                                -12384|                         -35467|            23083|                 3|
|2014|                                -25411|                          -6899|           -18512|                 4|
|2018|                                  5860|                         -12384|            18244|                 5|
|2002|                                  1128|                          19056|   

Задание 2: Категоризация годов по уровню чистой прибыли и агрегация
В задании Вам нужно категоризировать данные на основе условий и затем агрегировать другие показатели по этим категориям.

Создайте новую колонку категория_прибыли, которая будет содержать следующие значения на основе чистая_прибыль_отчётного_года_млн_руб:
* "Высокоприбыльный год": если чистая прибыль > 5000 млн руб.
* "Среднеприбыльный год": если чистая прибыль от 1000 до 5000 млн руб. (включительно)
* "Низкоприбыльный год": если чистая прибыль от 0 до 1000 млн руб. (исключая 0)
* "Убыточный год": если чистая прибыль < 0
* "Нулевая прибыль": если чистая прибыль = 0

Сгруппируйте данные по новой колонке категория_прибыли и для каждой категории рассчитайте:

* количество_лет: Сколько лет попадает в каждую категорию.
* средняя_общая_реализация_тыс_шт: Среднее количество проданных автомобилей (сумма внутренней и внешней реализации в тысячах штук).

Выведите полученные результаты агрегации, отсортировав их по количеству лет, по уменьшению .

In [72]:
df_auto_category = (
    df_auto_net_profit
    .withColumn(
        'категория_прибыли',
        F.when(
            F.col('чистая_прибыль_отчётного_года(млн_руб)') == 0,
            'Нулевая прибыль'
        )
        .when(
            F.col('чистая_прибыль_отчётного_года(млн_руб)') < 0,
            'Убыточный год'
        )
        .when(
            F.col('чистая_прибыль_отчётного_года(млн_руб)') > 5000,
            'Высокоприбыльный год'
        )
        .when(
            (F.col('чистая_прибыль_отчётного_года(млн_руб)') >= 1000) &
            (F.col('чистая_прибыль_отчётного_года(млн_руб)') <= 5000),
            'Среднеприбыльный год'
        )
        .when(
            (F.col('чистая_прибыль_отчётного_года(млн_руб)') > 0) &
            (F.col('чистая_прибыль_отчётного_года(млн_руб)') < 1000),
            'Низкоприбыльный год'
        )
        .otherwise('Неопределено')
    )
)

df_auto_category = (
    df_auto_net_profit
    .withColumn(
        'категория_прибыли',
        F.when(
            F.col('чистая_прибыль_отчётного_года(млн_руб)') == 0,
            'Нулевая прибыль'
        )
        .when(
            F.col('чистая_прибыль_отчётного_года(млн_руб)') < 0,
            'Убыточный год'
        )
        .when(
            F.col('чистая_прибыль_отчётного_года(млн_руб)') > 5000,
            'Высокоприбыльный год'
        )
        .when(
            (F.col('чистая_прибыль_отчётного_года(млн_руб)') >= 1000) &
            (F.col('чистая_прибыль_отчётного_года(млн_руб)') <= 5000),
            'Среднеприбыльный год'
        )
        .when(
            (F.col('чистая_прибыль_отчётного_года(млн_руб)') > 0) &
            (F.col('чистая_прибыль_отчётного_года(млн_руб)') < 1000),
            'Низкоприбыльный год'
        )
        .otherwise('Неопределено')
    )
)

df_auto_category.select(
    'год',
    'чистая_прибыль_отчётного_года(млн_руб)',
    'категория_прибыли'
).show()

+----+--------------------------------------+--------------------+
| год|чистая_прибыль_отчётного_года(млн_руб)|   категория_прибыли|
+----+--------------------------------------+--------------------+
|1999|                                  1021|Среднеприбыльный год|
|2000|                                  4554|Среднеприбыльный год|
|2001|                                 19056|Высокоприбыльный год|
|2002|                                  1128|Среднеприбыльный год|
|2003|                                  2951|Среднеприбыльный год|
|2004|                                  4575|Среднеприбыльный год|
|2005|                                  1400|Среднеприбыльный год|
|2006|                                  2512|Среднеприбыльный год|
|2007|                                  3951|Среднеприбыльный год|
|2008|                                 -6684|       Убыточный год|
|2009|                                -38500|       Убыточный год|
|2010|                                  1500|Среднеприбыльный 

In [90]:
df_groupby = df_auto_category \
              .groupby('категория_прибыли') \
              .agg(
                  F.count('год').alias('количество_лет'),
                  F.avg(F.col('реализация_(внутренний_рынок:_тыс)') + F.col('реализация_(внешний_рынок:_тыс)')).alias('средняя_общая_реализация_тыс_шт')
              ).withColumn(
        'средняя_общая_реализация_тыс_шт',
        F.round(F.col('средняя_общая_реализация_тыс_шт'), 2)
    ).orderBy('количество_лет', ascending=False)


df_groupby.show()

+--------------------+--------------+-------------------------------+
|   категория_прибыли|количество_лет|средняя_общая_реализация_тыс_шт|
+--------------------+--------------+-------------------------------+
|Среднеприбыльный год|            11|                         662.48|
|       Убыточный год|             8|                         412.63|
| Низкоприбыльный год|             3|                         394.17|
|Высокоприбыльный год|             2|                          582.0|
+--------------------+--------------+-------------------------------+



In [91]:
spark.stop()

# Анализ данных вселенной "Звездных Войн"

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
          .appName('ZV') \
          .getOrCreate()

In [49]:
df_zv_species = spark.read.parquet('/content/drive/MyDrive/py_spark_data/species.parquet')
df_zv_species.printSchema()
df_zv_species = df_zv_species.dropDuplicates(subset=["name","classification", "designation", "average_height"])
df_zv_species.show(3)

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- classification: string (nullable = true)
 |-- designation: string (nullable = true)
 |-- average_height: double (nullable = true)
 |-- skin_colors: string (nullable = true)
 |-- hair_colors: string (nullable = true)
 |-- eye_colors: string (nullable = true)
 |-- average_lifespan: double (nullable = true)
 |-- language: string (nullable = true)
 |-- homeworld: string (nullable = true)

+---+-----------+--------------+-----------+--------------+-----------+-----------+----------+----------------+---------+---------+
| id|       name|classification|designation|average_height|skin_colors|hair_colors|eye_colors|average_lifespan| language|homeworld|
+---+-----------+--------------+-----------+--------------+-----------+-----------+----------+----------------+---------+---------+
| 20| Chadra-Fan|        Mammal|   Sentient|           1.0|      Brown|      Brown|     Black|            50.0|Chadrafan|     Chad|
| 17| 

In [34]:
df_zv_species.drop_duplicates().count()


40

In [83]:
df_zv_organizations = spark.read.parquet('/content/drive/MyDrive/py_spark_data/organizations.parquet')
df_zv_organizations.printSchema()
df_zv_organizations = df_zv_organizations.drop_duplicates()
df_zv_organizations.show(3)

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- founded: long (nullable = true)
 |-- dissolved: long (nullable = true)
 |-- leader: string (nullable = true)
 |-- members: string (nullable = true)
 |-- affiliation: string (nullable = true)
 |-- description: string (nullable = true)
 |-- films: string (nullable = true)

+---+----------------+-------+---------+----------------+--------------------+-----------------+--------------------+--------------------+
| id|            name|founded|dissolved|          leader|             members|      affiliation|         description|               films|
+---+----------------+-------+---------+----------------+--------------------+-----------------+--------------------+--------------------+
|  6|Trade Federation|   -350|        0|     Nute Gunray|Rune Haako, Lott Dod|      Separatists|A commercial trad...|The Phantom Menac...|
|  1|      Jedi Order| -25000|       19|Yoda, Mace Windu|Obi-Wan Kenobi, A...|Galactic Republi

In [65]:
df_zv_characters = spark.read.parquet('/content/drive/MyDrive/py_spark_data/characters.parquet')
df_zv_characters.printSchema()
df_zv_characters = df_zv_characters.dropDuplicates(subset=["name", "species", "homeworld", "year_born"])
df_zv_characters.show(3)

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- species: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- height: double (nullable = true)
 |-- weight: double (nullable = true)
 |-- hair_color: string (nullable = true)
 |-- eye_color: string (nullable = true)
 |-- skin_color: string (nullable = true)
 |-- year_born: double (nullable = true)
 |-- homeworld: string (nullable = true)
 |-- year_died: double (nullable = true)
 |-- description: string (nullable = true)

+---+--------------+------------+------+------+------+----------+---------+----------+---------+---------+---------+--------------------+
| id|          name|     species|gender|height|weight|hair_color|eye_color|skin_color|year_born|homeworld|year_died|         description|
+---+--------------+------------+------+------+------+----------+---------+----------+---------+---------+---------+--------------------+
| 29|  Aayla Secura|     Twi'lek|Female|  1.78|  55.0|      Blue|   

Задачи:

1. Распространенность видов и их классификаций: Посчитайте количество персонажей для каждого вида (characters.species), а также для каждой биологической классификации (species.classification). Для классификаций вам потребуется объединить таблицы characters и species.
* Вывод 1: название вида (species), количество персонажей (character_count). Отсортируйте по количеству персонажей по убыванию. Вывести 10 первых строк.
* Вывод 2: биологическая классификация (classification), общее количество персонажей (total_character_count). Отсортируйте по количеству персонажей по убыванию.

In [68]:
(df_zv_characters.groupBy("species")
    .agg(F.count("*").alias("character_count"))
    .orderBy(F.col("character_count").desc())
    .limit(10).show())

(df_zv_species.join(df_zv_characters, df_zv_species.name == df_zv_characters.species, "left")
    .groupBy(df_zv_species.classification).agg(F.count("*").alias("total_character_count"))
    .orderBy(F.col("total_character_count").desc())
    .show())

+------------+---------------+
|     species|character_count|
+------------+---------------+
|       Human|             47|
|       Droid|              4|
|     Twi'lek|              3|
| Dathomirian|              3|
|     Togruta|              2|
|     Unknown|              2|
|Mon Calamari|              2|
|    Clawdite|              1|
|       Chiss|              1|
|    Besalisk|              1|
+------------+---------------+

+--------------+---------------------+
|classification|total_character_count|
+--------------+---------------------+
|        Mammal|                   73|
|     Amphibian|                    6|
|     Reptilian|                    5|
|    Artificial|                    4|
|        Hybrid|                    3|
|       Unknown|                    1|
|     Insectoid|                    1|
|     Gastropod|                    1|
+--------------+---------------------+



Средний рост по классификации видов:  Рассчитайте средний рост (average_height) для каждой classification в таблице species.  Округлите результат до одного знака после запятой.​​​

Вывод: классификация (classification), средний рост (average_height_class). Отсортируйте по среднему росту, по убыванию.

In [77]:
avg_height_class = df_zv_species \
                    .groupBy('classification') \
                    .agg(F.round(F.avg('average_height'), 1).alias('avg_height_class')) \
                    .orderBy(F.col('avg_height_class').desc())
avg_height_class.show()

+--------------+----------------+
|classification|avg_height_class|
+--------------+----------------+
|     Gastropod|             3.9|
|     Amphibian|             1.9|
|     Insectoid|             1.8|
|     Reptilian|             1.8|
|        Mammal|             1.7|
|        Hybrid|             1.7|
|       Unknown|             0.7|
|    Artificial|            NULL|
+--------------+----------------+



Члены Ордена Джедаев и Ситхов: Получите список всех лидеров и членов организаций "Jedi Order" и "Sith Order". Предварительно необходимо преобразовать столбцы leader и members в формат Array.

Вывод: название организации (name), имя члена организации (name_member). Отсортируйте по  названию организации.

In [84]:
df_zv_organizations = df_zv_organizations.withColumn(
    'leader',
    F.split(F.col('leader'), ',')
).withColumn(
    'members',
    F.split(F.col('members'), ',')
)

In [105]:
members = (df_zv_organizations
           .select("name", F.explode_outer("leader").alias("name_member"))
           .filter(F.col("name").isin("Jedi Order", "Sith Order"))
    .union
           (df_zv_organizations
            .select("name", F.explode_outer("members").alias("name_member"))
            .filter(F.col("name").isin("Jedi Order", "Sith Order")))
            )
members.show()

+----------+-----------------+
|      name|      name_member|
+----------+-----------------+
|Jedi Order|             Yoda|
|Jedi Order|       Mace Windu|
|Sith Order|    Darth Sidious|
|Jedi Order|   Obi-Wan Kenobi|
|Jedi Order| Anakin Skywalker|
|Jedi Order|   Luke Skywalker|
|Sith Order|      Darth Vader|
|Sith Order|       Darth Maul|
|Sith Order|    Darth Tyranus|
+----------+-----------------+



Топ-5 старейших и топ-5 самых юных персонажей: Определите 5 персонажей имеющих максимальное и минимальное значение year_born. Исключите персонажей в отсутствующим возрастом.

1. Вывод 1: 5 старейших персонажей. Имя персонажа (name), год рождения (year_born).

2. Вывод 2: 5 самых юных персонажей. Имя персонажа (name), год рождения (year_born).

In [108]:
df_zv_characters_top5_max = df_zv_characters.filter(df_zv_characters.year_born.isNotNull()).orderBy(df_zv_characters.year_born.desc()).limit(5)
df_zv_characters_top5_min = df_zv_characters.filter(df_zv_characters.year_born.isNotNull()).orderBy(df_zv_characters.year_born.asc()).limit(5)
df_zv_characters_top5_max.select('name', 'year_born').show()
df_zv_characters_top5_min.select('name', 'year_born').show()

+--------------+---------+
|          name|year_born|
+--------------+---------+
|    Maz Kanata|    973.0|
|          Yoda|    896.0|
|Jabba the Hutt|    600.0|
|     Chewbacca|    200.0|
|         C-3PO|    112.0|
+--------------+---------+

+-----------+---------+
|       name|year_born|
+-----------+---------+
|General Hux|      0.0|
|Poe Dameron|      2.0|
|   Kylo Ren|      5.0|
|  Rose Tico|     11.0|
|       Finn|     11.0|
+-----------+---------+



Персонажи с экстремальным индексом массы тела: Найдите 5 персонажей с максимальным и 5 персонажей с минимальным индексом массы тела (ИМТ). Раcсчитайте ИМТ: weight / height**2. Исключите NULL из вывода.

Вывод 1: 5 персонажей с максимальным ИМТ

Вывод 2: 5 персонажей с минимальным ИМТ

In [117]:
df_zv_characters_top5_imt = df_zv_characters.withColumn(
    'imt',
    F.round(F.col('weight') / F.col('height')**2, 2)
)
df_zv_characters_top5_imt.select('name', 'imt').filter(df_zv_characters_top5_imt.imt.isNotNull()).sort(df_zv_characters_top5_imt.imt.asc()).limit(5).show()
df_zv_characters_top5_imt.select('name', 'imt').filter(df_zv_characters_top5_imt.imt.isNotNull()).sort(df_zv_characters_top5_imt.imt.desc()).limit(5).show()


+-------------+-----+
|         name|  imt|
+-------------+-----+
|  Ahsoka Tano|15.56|
|Padmé Amidala|16.53|
|Jar Jar Binks|17.18|
|        Qi'ra| 17.3|
|Hera Syndulla|17.36|
+-------------+-----+

+--------------+-----+
|          name|  imt|
+--------------+-----+
|Jabba the Hutt|89.28|
|    Pong Krell| 48.0|
|          Yoda|39.03|
|   Darth Vader|33.33|
|       Sebulba|31.89|
+--------------+-----+



In [118]:
spark.stop()

# Анализ пользовательских сессий

# Анализ пользовательских сессий. Netflix